In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import torch
import scipy.sparse

import GenKI as gk
from GenKI.preprocesing import build_adata
from GenKI.dataLoader import DataLoader
from GenKI.train import VGAE_trainer
from GenKI import utils

%load_ext autoreload
%autoreload 2
sc.settings.verbosity = 0

In [ ]:
os.chdir('/data/fuyq/GenKI/result/gene_p/')

In [ ]:
prefix = "/data/fuyq/GenKI/data/WT_new/"

counts = sc.read_mtx(prefix + "matrix.mtx").T  
barcodes = pd.read_csv(prefix + "barcodes.txt", header=None)[0].values
genes = pd.read_csv(prefix + "geneNames.txt", header=None)[0].values

adata = sc.AnnData(X=counts)
adata.obs_names = barcodes
adata.var_names = genes

In [ ]:
original_counts = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["norm"] = adata.X

sc.pp.highly_variable_genes(adata, n_top_genes=3000)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)

highly_variable_mask = adata.var['highly_variable'].values
print("Number of highly variable genes:", adata.var['highly_variable'].sum())

In [ ]:
highly_variable_genes = adata.var_names[adata.var['highly_variable']]

with open('/data/fuyq/GenKI/data/tf_mm_new.txt', 'r') as f:
    tf_genes = f.read().splitlines()

intersection_genes = set(highly_variable_genes).intersection(tf_genes)
print(f"交集基因的数量：{len(intersection_genes)}")

intersection_df = pd.DataFrame(list(intersection_genes), columns=['Gene'])
intersection_df.to_csv('/data/fuyq/GenKI/result/highly_variable_genes.csv', index=False)

In [ ]:
if isinstance(adata.X, np.ndarray):
    import scipy.sparse
    adata.X = scipy.sparse.csr_matrix(adata.X)

In [ ]:
for gene in sorted(intersection_genes):
    print(f"Processing gene: {gene}")

    target_gene = [gene]

    data_wrapper = DataLoader(
        adata,  # adata object
        target_gene=target_gene,  # KO gene name
        target_cell=None,  # obsname for cell type, if none use all
        obs_label="ident",  # colname for genes
        GRN_file_dir="/data/fuyq/GenKI/result/GRN/",  # folder name for GRNs
        rebuild_GRN=False,  # whether build GRN by pcNet
        pcNet_name="Hmgb2_Net",  # GRN file name
        verbose=True,  # whether verbose
        n_cpus=8  # multiprocessing
    )

    data_wt = data_wrapper.load_data()  # Load WT data
    data_ko = data_wrapper.load_kodata()  # Load KO data

    # Hyperparameters
    hyperparams = {
        "epochs": 100,
        "lr": 7e-4,
        "beta": 1e-4,
        "seed": 8096
    }
    log_dir = None

    # Initialize the VGAE_trainer
    sensei = VGAE_trainer(
        data_wt,
        epochs=hyperparams["epochs"],
        lr=hyperparams["lr"],
        log_dir=log_dir,
        beta=hyperparams["beta"],
        seed=hyperparams["seed"],
        verbose=False
    )

    # Train the model
    sensei.train()

    device = torch.device("cpu")  
    sensei.model.to(device)       

    # Get latent variables for WT and KO data
    z_mu_wt, z_std_wt = sensei.get_latent_vars(data_wt)
    z_mu_ko, z_std_ko = sensei.get_latent_vars(data_ko)

    
    # Calculate the distance between KO and WT
    dis = gk.utils.get_distance(z_mu_ko, z_std_ko, z_mu_wt, z_std_wt, by="KL")
    print(dis.shape)

    # Get gene ranking
    res_raw = utils.get_generank(data_wt, dis, rank=True)
    
    # Save results to a CSV file
    res_raw.to_csv(f'/data/fuyq/GenKI/result/gene_list/{gene}_results.csv', index=True)

    # Perform permutation test
    null = sensei.pmt(data_ko, n=100, by="KL")

    # Get gene ranking with null distribution
    res = utils.get_generank(data_wt, dis, null, save_significant_as=f'{gene}')
    
    # Optionally, you can also save the second result
    res.to_csv(f'/data/fuyq/GenKI/result/gene_list/{gene}_significant_results.csv', index=True)

    print(f"Finished processing {gene}")